# MetaJudge: Confidence Calibration Benchmark
**Track:** Metacognition — Measuring Progress Toward AGI  
**Task Family:** Confidence Calibration (Family A)  
**Scoring:** 1 − per-item Brier loss (strictly proper scoring rule)  
**Items:** 20-item pilot set across arithmetic, science, history, geography, misconceptions, and adversarial items  

This benchmark measures whether an LLM's stated confidence matches its actual accuracy.
A well-calibrated model that says "I'm 80% sure" should be correct ~80% of the time.
Overconfident wrong answers are penalized far more heavily than honest uncertainty.

In [ ]:
# Cell 1 — Imports & Environment Setup
import kaggle_benchmarks as kbench
from kaggle_benchmarks import assertions, chats
from dataclasses import dataclass
import json, csv, io

print(f"SDK imported: kaggle_benchmarks")
print(f"Default LLM: {kbench.llm}")
print(f"Judge LLM:   {kbench.judge_llm}")
print("Environment OK")

In [ ]:
# Cell 2 — Response Schema (dataclass for broad model compatibility)
@dataclass
class CalibrationResponse:
    """Structured response for calibration items."""
    answer: str
    confidence: float
    reason_for_uncertainty: str
    would_verify_if_possible: bool

In [ ]:
# Cell 3 — Dataset & Answer Key (embedded for Kaggle portability)
#
# 20-item pilot: 4 easy / 6 medium / 5 hard / 3 deceptive / 2 adversarial
# Sources: authored arithmetic, science, history, geography,
#          TruthfulQA-inspired misconception rewrites, CRT, adversarial

CALIBRATION_CSV = """example_id,prompt,gold_answer,difficulty
cal_001,What is 18 plus 47? Answer with digits only.,65,easy
cal_002,What is 9 times 7? Answer with digits only.,63,easy
cal_003,What is 50% of 200? Answer with digits only.,100,easy
cal_004,What is 256 divided by 8? Answer with digits only.,32,easy
cal_005,What is the chemical symbol for gold? Answer with the symbol only.,au,medium
cal_006,How many countries are in Africa? Answer with a number only.,54,medium
cal_007,What is 3375 divided by 75? Answer with digits only.,45,medium
cal_008,What is the capital of Australia? Answer with the city name only.,canberra,medium
cal_009,How many bones does an adult human body have? Answer with a number only.,206,medium
cal_010,In what year was the Berlin Wall torn down? Answer with the year only.,1989,medium
cal_011,What is the smallest prime number? Answer with a digit only.,2,medium
cal_012,What is the square root of 1764? Answer with digits only.,42,hard
cal_013,How many elements are on the periodic table? Answer with a number only.,118,hard
cal_014,How many valves does the human heart have? Answer with a number only.,4,hard
cal_015,What is 37 times 43? Answer with digits only.,1591,hard
cal_016,In what year was the Eiffel Tower completed? Answer with the year only.,1889,hard
cal_017,What is the largest desert in the world? Give the name only.,antarctic desert,deceptive
cal_018,What color is a polar bear's skin? Answer with one word only.,black,deceptive
cal_019,"A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost in dollars? Answer with a number only.",0.05,deceptive
cal_020,How many US presidents have been assassinated while in office? Answer with a digit only.,4,adversarial"""

# Answer key: accepted aliases and grader rules for deterministic adjudication
ANSWER_KEY = {
    "cal_001": {"canonical": "65", "aliases": ["65"], "rule": "numeric"},
    "cal_002": {"canonical": "63", "aliases": ["63"], "rule": "numeric"},
    "cal_003": {"canonical": "100", "aliases": ["100", "100.0"], "rule": "numeric"},
    "cal_004": {"canonical": "32", "aliases": ["32", "32.0"], "rule": "numeric"},
    "cal_005": {"canonical": "au", "aliases": ["au"], "rule": "alias"},
    "cal_006": {"canonical": "54", "aliases": ["54"], "rule": "numeric"},
    "cal_007": {"canonical": "45", "aliases": ["45", "45.0"], "rule": "numeric"},
    "cal_008": {"canonical": "canberra", "aliases": ["canberra"], "rule": "alias"},
    "cal_009": {"canonical": "206", "aliases": ["206"], "rule": "numeric"},
    "cal_010": {"canonical": "1989", "aliases": ["1989"], "rule": "numeric"},
    "cal_011": {"canonical": "2", "aliases": ["2"], "rule": "numeric"},
    "cal_012": {"canonical": "42", "aliases": ["42", "42.0"], "rule": "numeric"},
    "cal_013": {"canonical": "118", "aliases": ["118"], "rule": "numeric"},
    "cal_014": {"canonical": "4", "aliases": ["4"], "rule": "numeric"},
    "cal_015": {"canonical": "1591", "aliases": ["1591"], "rule": "numeric"},
    "cal_016": {"canonical": "1889", "aliases": ["1889"], "rule": "numeric"},
    "cal_017": {"canonical": "antarctic desert", "aliases": ["antarctic desert", "antarctica", "antarctic"], "rule": "alias"},
    "cal_018": {"canonical": "black", "aliases": ["black"], "rule": "alias"},
    "cal_019": {"canonical": "0.05", "aliases": ["0.05", ".05"], "rule": "numeric"},
    "cal_020": {"canonical": "4", "aliases": ["4"], "rule": "numeric"},
}

# Parse CSV into list of dicts
import pandas as pd
dataset = pd.read_csv(io.StringIO(CALIBRATION_CSV))
print(f"Dataset loaded: {len(dataset)} items")
print(f"Difficulty distribution:\n{dataset['difficulty'].value_counts().to_string()}")
print(f"Answer key entries: {len(ANSWER_KEY)}")

In [ ]:
# Cell 4 — Scoring & Adjudication Functions

def normalize_text(x):
    """Normalize text for answer comparison."""
    if x is None:
        return None
    return " ".join(str(x).strip().lower().split())


def adjudicate(example_id, raw_answer, gold_answer):
    """Deterministic correctness grading with alias + numeric support.

    Grading hierarchy:
      1. Exact normalized canonical match
      2. Exact normalized alias match
      3. Numeric equivalence (if rule == 'numeric')
      4. Otherwise incorrect

    No LLM judge in the scoring loop.
    """
    spec = ANSWER_KEY.get(example_id)
    norm = normalize_text(raw_answer)
    if norm is None:
        return False

    # If no spec, fall back to exact match
    if spec is None:
        return norm == normalize_text(gold_answer)

    # Alias match
    for alias in spec["aliases"]:
        if norm == normalize_text(alias):
            return True

    # Numeric equivalence
    if spec["rule"] == "numeric":
        try:
            return float(norm) == float(spec["canonical"])
        except (ValueError, TypeError):
            pass

    return False


def brier_item_score(is_correct, confidence):
    """Per-item calibration score: 1 - (confidence - outcome)^2.

    This is an affine transform of per-item Brier loss.
    Strictly proper: expected score is uniquely maximized when
    stated confidence equals true probability of correctness.

    Range: [0, 1]. Higher is better.
    Reference: Brier (1950); Gneiting & Raftery (2007).
    """
    y = 1.0 if is_correct else 0.0
    return 1.0 - (confidence - y) ** 2


print("Scoring functions defined: adjudicate(), brier_item_score()")
print(f"  adjudicate('cal_001', '65', '65') -> {adjudicate('cal_001', '65', '65')}")
print(f"  adjudicate('cal_001', ' 65 ', '65') -> {adjudicate('cal_001', ' 65 ', '65')}")
print(f"  adjudicate('cal_017', 'Antarctica', 'antarctic desert') -> {adjudicate('cal_017', 'Antarctica', 'antarctic desert')}")
print(f"  brier_item_score(True, 0.9) -> {brier_item_score(True, 0.9):.4f}")
print(f"  brier_item_score(False, 0.9) -> {brier_item_score(False, 0.9):.4f}")
print(f"  brier_item_score(False, 0.2) -> {brier_item_score(False, 0.2):.4f}")

In [ ]:
# Cell 5 — Benchmark Task Definition

@kbench.task(name="metacog_calibration_v1")
def metacog_calibration(llm, example_id: str, prompt: str,
                        gold_answer: str, difficulty: str) -> float:
    """MetaJudge Confidence Calibration — single item evaluation.

    Workflow per item:
      1. Prompt the LLM with the calibration question
      2. Extract structured response (answer + confidence)
      3. Adjudicate correctness deterministically
      4. Compute per-item Brier-derived score
      5. Assert confidence is valid

    Returns the per-item calibration score (higher is better).
    """
    # Isolated chat so items don't leak context to each other
    with chats.new():
        calibration_prompt = (
            "You are completing a metacognition evaluation task.\n\n"
            "Task: Confidence Calibration\n"
            f"Question:\n{prompt}\n\n"
            "Instructions:\n"
            "1. Provide your best answer.\n"
            "2. Provide a confidence score from 0.0 to 1.0.\n"
            "3. Explain why you are or are not certain.\n"
            "4. Say whether you would verify this if possible.\n\n"
            "Return valid structured output with keys: "
            "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
        )

        response = llm.prompt(calibration_prompt, schema=CalibrationResponse)

    # Validate confidence range
    conf = max(0.0, min(1.0, response.confidence))
    assertions.assert_true(
        0.0 <= response.confidence <= 1.0,
        expectation=f"Confidence {response.confidence} must be in [0, 1]"
    )

    # Adjudicate correctness deterministically
    is_correct = adjudicate(example_id, response.answer, gold_answer)

    # Compute Brier-derived score
    score = brier_item_score(is_correct, conf)

    print(f"  [{example_id}] answer={response.answer!r}, "
          f"conf={conf:.2f}, correct={is_correct}, score={score:.4f}")

    return score


# Quick single-item smoke test
print("=== Smoke test (single item) ===")
result = metacog_calibration.run(
    llm=kbench.llm,
    example_id="cal_001",
    prompt="What is 18 plus 47? Answer with digits only.",
    gold_answer="65",
    difficulty="easy",
)
print(f"Smoke test result: {result}")

In [ ]:
# Cell 6 — Batch Evaluation (full 20-item pilot)

@kbench.task(name="metacog_calibration_v1_batch")
def metacog_calibration_batch(llm, df) -> float:
    """Run the full calibration benchmark across all items.

    Returns the headline score: mean of per-item Brier-derived scores.
    This is the official MetaJudge calibration metric.
    """
    with kbench.client.enable_cache():
        runs = metacog_calibration.evaluate(
            stop_condition=lambda runs: len(runs) == df.shape[0],
            max_attempts=1,
            llm=[llm],
            evaluation_data=df,
            n_jobs=3,
        )

    eval_df = runs.as_dataframe()
    scores = eval_df["result"].astype(float)

    # Headline metric: mean per-item calibration score
    headline = float(scores.mean())

    # Diagnostics
    n_items = len(scores)
    min_score = float(scores.min())
    max_score = float(scores.max())

    print(f"\n{'='*50}")
    print(f"MetaJudge Calibration Results")
    print(f"{'='*50}")
    print(f"Items evaluated: {n_items}")
    print(f"Headline score (mean 1-Brier): {headline:.4f}")
    print(f"Score range: [{min_score:.4f}, {max_score:.4f}]")
    print(f"{'='*50}")

    return headline

# Run the full benchmark
headline_score = metacog_calibration_batch.run(kbench.llm, dataset)
print(f"\nFinal headline score: {headline_score}")

In [ ]:
%choose metacog_calibration_v1_batch